# 🧠 Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

---
### 🛠️ What You Need to Implement
- Agent logic
- Conditional routing
- Tool integration
- Basic error handling

### 🚀 Bonus
- Improve routing
- Add logging
- Add more tools

In [1]:
import ast
import operator
import logging
import re

## Configure Logging

In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s - %(message)s"
)

## Tool 1 : Calculator

In [3]:
allowed_operations = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.USub: operator.neg
}

In [4]:
def evaluate(node):

    if isinstance(node, ast.Constant):
        return node.value

    if isinstance(node, ast.Num):
        return node.n

    if isinstance(node, ast.BinOp):
        left = evaluate(node.left)
        right = evaluate(node.right)

        operation = allowed_operations.get(type(node.op))

        if operation is None:
            raise ValueError("Unsupported operator")

        return operation(left, right)

    if isinstance(node, ast.UnaryOp):
        operation = allowed_operations.get(type(node.op))

        if operation is None:
            raise ValueError("Unsupported unary operator")

        return operation(evaluate(node.operand))

    raise ValueError("Invalid mathematical expression")

In [5]:
def calculator(expression):

    try:
        parsed = ast.parse(expression, mode="eval")
        answer = evaluate(parsed.body)
        return str(answer)

    except Exception:
        return "Error in calculation"

## Tool 2 : Keyword Extractor

In [6]:
def extract_keywords(text):

    try:

        words = re.findall(r"\b[a-zA-Z]{5,}\b", text.lower())

        unique_words = []

        for word in words:

            if word not in unique_words:
                unique_words.append(word)

        return unique_words[:5]

    except Exception:

        return []

##  Tool 3 : Word Counter

In [7]:
def word_counter(text):

    total = len(text.split())

    return {
        "total_words": total
    }

## 🤖 Agent Logic and Conditional Routing

The agent analyzes the user query and routes it to the appropriate tool based on its intent.

- Mathematical queries → Calculator Tool
- Keyword queries → Keyword Extraction Tool
- Word counting queries → Word Counter Tool
- Other queries → General Response
- Invalid inputs → Error Response

In [8]:
def get_math_expression(query):
    """
    Extract the mathematical expression from a calculation query.
    """
    expression = re.sub(
        r"^\s*calculate\s*",
        "",
        query,
        flags=re.IGNORECASE
    ).strip()

    return expression

In [9]:
def get_keyword_text(query):
    """
    Extract the text that needs keyword processing.
    """
    text = re.sub(
        r"^\s*(extract\s+)?keywords?\s*(from)?\s*",
        "",
        query,
        flags=re.IGNORECASE
    ).strip()

    return text

## 🤖 Single-Agent Router

The agent uses conditional routing to identify the user's intent.

It checks the normalized query and selects the appropriate tool. Each response is returned in a structured format containing a `type` and `result`.

In [10]:
def agent(query: str):

    # Validate input
    if not isinstance(query, str) or not query.strip():
        logging.error("Invalid or empty query received.")

        return {
            "type": "error",
            "result": "Please enter a valid query."
        }

    normalized_query = query.strip().lower()

    logging.info(f"Processing query: {query}")

    try:

        # Route mathematical queries
        if "calculate" in normalized_query:

            logging.info("Routing to Calculator Tool.")

            expression = get_math_expression(query)

            if not expression:
                return {
                    "type": "error",
                    "result": "No mathematical expression was provided."
                }

            result = calculator(expression)

            if result == "Error in calculation":
                return {
                    "type": "error",
                    "result": result
                }

            return {
                "type": "calculation",
                "result": result
            }

        # Route keyword extraction queries
        elif "keyword" in normalized_query:

            logging.info("Routing to Keyword Extraction Tool.")

            text = get_keyword_text(query)

            if not text:
                return {
                    "type": "error",
                    "result": "No text was provided for keyword extraction."
                }

            result = extract_keywords(text)

            return {
                "type": "keywords",
                "result": result
            }

        # Bonus word counter route
        elif "count words" in normalized_query or "word count" in normalized_query:

            logging.info("Routing to Word Counter Tool.")

            text = re.sub(
                r"^\s*(count\s+words\s+in|word\s+count\s+for)\s*",
                "",
                query,
                flags=re.IGNORECASE
            ).strip()

            if not text:
                return {
                    "type": "error",
                    "result": "No text was provided for word counting."
                }

            result = word_counter(text)

            return {
                "type": "word_count",
                "result": result
            }

        # General response
        else:

            logging.info("Routing to General Response.")

            return {
                "type": "general",
                "result": f"You asked: {query}"
            }

    except Exception as e:

        logging.error(f"Agent error: {e}")

        return {
            "type": "error",
            "result": "An error occurred while processing the query."
        }

##  Expected Output Format

Every agent response follows a structured dictionary format:

```python
{
    "type": "...",
    "result": ...
}
```

In [11]:
queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?"
]

for q in queries:

    print("Query:", q)
    print("Response:", agent(q))
    print("-" * 50)

Query: Calculate 20 + 5
Response: {'type': 'calculation', 'result': '25'}
--------------------------------------------------
Query: Extract keywords from Artificial Intelligence is transforming industries
Response: {'type': 'keywords', 'result': ['artificial', 'intelligence', 'transforming', 'industries']}
--------------------------------------------------
Query: What is machine learning?
Response: {'type': 'general', 'result': 'You asked: What is machine learning?'}
--------------------------------------------------


/tmp/ipykernel_2291/1006293139.py:6: DeprecationWarning: ast.Num is deprecated and will be removed in Python 3.14; use ast.Constant instead
  if isinstance(node, ast.Num):


## 🔍 Error Handling and Validation Tests

The following cases check how the agent behaves when it receives invalid, incomplete, or unsupported inputs.

In [12]:
test_queries = [
    "",
    "Calculate",
    "Calculate 10 / 0",
    "Extract keywords from",
    "Hello agent",
    "Count words in Artificial Intelligence is useful"
]

for q in test_queries:

    print("Query:", repr(q))
    print("Response:", agent(q))
    print("-" * 60)

ERROR:root:Invalid or empty query received.


Query: ''
Response: {'type': 'error', 'result': 'Please enter a valid query.'}
------------------------------------------------------------
Query: 'Calculate'
Response: {'type': 'error', 'result': 'No mathematical expression was provided.'}
------------------------------------------------------------
Query: 'Calculate 10 / 0'
Response: {'type': 'error', 'result': 'Error in calculation'}
------------------------------------------------------------
Query: 'Extract keywords from'
Response: {'type': 'error', 'result': 'No text was provided for keyword extraction.'}
------------------------------------------------------------
Query: 'Hello agent'
Response: {'type': 'general', 'result': 'You asked: Hello agent'}
------------------------------------------------------------
Query: 'Count words in Artificial Intelligence is useful'
Response: {'type': 'word_count', 'result': {'total_words': 4}}
------------------------------------------------------------


/tmp/ipykernel_2291/1006293139.py:6: DeprecationWarning: ast.Num is deprecated and will be removed in Python 3.14; use ast.Constant instead
  if isinstance(node, ast.Num):


##  Additional Routing Tests

These examples verify that the agent can route different variations of user requests to the appropriate tools.

In [13]:
additional_queries = [
    "calculate 100 - 35",
    "Calculate 8 * 7",
    "keywords from Data Science and Artificial Intelligence",
    "Count words in machine learning is changing technology",
    "Tell me something about Python"
]

for q in additional_queries:

    print("Query:", q)
    print("Response:", agent(q))
    print("=" * 60)

Query: calculate 100 - 35
Response: {'type': 'calculation', 'result': '65'}
Query: Calculate 8 * 7
Response: {'type': 'calculation', 'result': '56'}
Query: keywords from Data Science and Artificial Intelligence
Response: {'type': 'keywords', 'result': ['science', 'artificial', 'intelligence']}
Query: Count words in machine learning is changing technology
Response: {'type': 'word_count', 'result': {'total_words': 5}}
Query: Tell me something about Python
Response: {'type': 'general', 'result': 'You asked: Tell me something about Python'}


/tmp/ipykernel_2291/1006293139.py:6: DeprecationWarning: ast.Num is deprecated and will be removed in Python 3.14; use ast.Constant instead
  if isinstance(node, ast.Num):


##  Interactive Mode

The interactive mode allows users to continuously send queries to the agent. The session ends when the user enters `exit`.

In [14]:
while True:

    user_input = input("Enter query (type 'exit' to stop): ")

    if user_input.strip().lower() == "exit":
        print("Agent session ended.")
        break

    response = agent(user_input)

    print("Response:", response)

Enter query (type 'exit' to stop): Calculate 45 * 3


/tmp/ipykernel_2291/1006293139.py:6: DeprecationWarning: ast.Num is deprecated and will be removed in Python 3.14; use ast.Constant instead
  if isinstance(node, ast.Num):


Response: {'type': 'calculation', 'result': '135'}
Enter query (type 'exit' to stop): Extract keywords from Deep Learning is used in computer vision
Response: {'type': 'keywords', 'result': ['learning', 'computer', 'vision']}
Enter query (type 'exit' to stop): What is artificial intelligence?
Response: {'type': 'general', 'result': 'You asked: What is artificial intelligence?'}
Enter query (type 'exit' to stop): Count words in Data Science is an interdisciplinary field
Response: {'type': 'word_count', 'result': {'total_words': 6}}
Enter query (type 'exit' to stop): exit
Agent session ended.


##  Agent Workflow

The Single-Agent Smart Assistant follows an intent-based routing process.

1. The user provides a query.
2. The agent validates and normalizes the input.
3. The query is checked to identify its intent.
4. The appropriate tool is selected based on the query.
5. The selected tool processes the request.
6. The result is returned in a structured format.
7. Queries that do not match any specific tool are handled by the general response.
8. Invalid inputs are handled using error responses.

##  Agent Pipeline

```text
                     User Query
                         │
                         ▼
                 Input Validation
                         │
                         ▼
                  Intent Detection
                         │
          ┌──────────────┼──────────────┐
          │              │              │
          ▼              ▼              ▼
      Calculate       Keywords      Word Count
          │              │              │
          ▼              ▼              ▼
     Calculator     Keyword Tool   Word Counter
          │              │              │
          └──────────────┼──────────────┘
                         │
                         ▼
                 Structured Response
                         │
                         ▼
                     JSON Output

              Other / General Queries
                         │
                         ▼
                  General Response

## Evaluation Summary

The agent was tested with different types of inputs to verify its routing and error-handling capabilities.

| Test Category | Example | Result |
|---|---|---|
| Mathematical Query | Calculate 20 + 5 | ✅ Passed |
| Invalid Calculation | Calculate 10 / 0 | ✅ Passed |
| Keyword Extraction | Extract keywords from AI text | ✅ Passed |
| Word Counting | Count words in a sentence | ✅ Passed |
| General Query | What is machine learning? | ✅ Passed |
| Empty Input | Empty query | ✅ Passed |
| Interactive Mode | Multiple user queries | ✅ Passed |

The tests show that the agent correctly routes queries to the appropriate tools and returns structured responses. Invalid and incomplete inputs are also handled without stopping the pipeline.

## Conclusion

The Single-Agent Smart Assistant successfully performs intent-based routing of user queries.

The system integrates multiple tools including a Calculator, Keyword Extractor, and Word Counter. Conditional routing allows the agent to select the appropriate tool based on the user's query.

The implementation also includes input validation, error handling, logging, structured responses, automated testing, and an interactive mode.

Overall, the project demonstrates how a single agent can coordinate different tools through a simple conditional pipeline.